[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPc/poisson_regression.ipynb)



# Poisson (count) regression with `gpc`



GPc port of [`GPy/Poisson regression tutorial.ipynb`](../GPy/Poisson%20regression%20tutorial.ipynb) (Alan Saul / Neil Lawrence), using the CIP-003A likelihood zoo.



Count observations \(Y \in \{0,1,2,\ldots\}\) with a log-Gaussian intensity (GP prior on \(\log \lambda\)) and a **Laplace** approximation:



```python

lik = gpc.likelihoods.Poisson()

m = gpc.models.GPRegression(X, Y, kernel=k, likelihood=lik, inference="laplace")

m.optimize()

mu_f, var_f = m.predict_noiseless(Xtest)  # latent log-intensity

```



Requires **`gpc>=0.1.0a10`**. Heteroscedastic / HMC / warped likelihoods remain out of subset.


In [ ]:
import importlib
import re
import subprocess
import sys


def _gpc_ver_tuple(v: str):
    """Parse ``0.1.0a10`` / ``0.1.0`` into a comparable tuple (final > any alpha)."""
    m = re.fullmatch(r"(\d+)\.(\d+)\.(\d+)(?:a(\d+))?", str(v))
    if not m:
        return (0, 0, 0, 0)
    maj, mir, pat, alpha = m.groups()
    pre = int(alpha) if alpha is not None else 10**9
    return (int(maj), int(mir), int(pat), pre)


_MIN = "0.1.0a10"
_need = _gpc_ver_tuple(_MIN)


def _ensure_gpc():
    try:
        import gpc as _gpc
        if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) >= _need:
            return _gpc
    except ImportError:
        pass
    # Prefer TestPyPI for pre-release alphas; real PyPI for dependencies (CIP-002C).
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--no-cache-dir",
            "--index-url",
            "https://test.pypi.org/simple/",
            "--extra-index-url",
            "https://pypi.org/simple/",
            f"gpc>={_MIN}",
        ]
    )
    if "gpc" in sys.modules:
        importlib.reload(sys.modules["gpc"])
    import gpc as _gpc
    if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) < _need:
        raise RuntimeError(
            f"need gpc>={_MIN}, got {getattr(_gpc, '__version__', '?')}"
        )
    return _gpc


gpc = _ensure_gpc()


In [ ]:
import os

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")



import numpy as np

import gpc

from matplotlib import pyplot as plt



try:

    from scipy import stats

except ImportError:

    import subprocess, sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scipy"])

    from scipy import stats


## Poisson mass function



As the rate grows, the Poisson PMF approaches a Gaussian of matching mean and variance. For small counts the discrete support and non-negativity matter — hence the log link + Laplace path rather than a plain Gaussian GP.


In [ ]:
K = np.arange(0, 30)

rates_plot = [1, 3, 8, 20]

for r in rates_plot:

    plt.plot(K, stats.poisson(r).pmf(K), label=f"rate = {r}")

plt.title("Poisson PMF for different rates")

plt.xlabel("Counts")

plt.ylabel("PMF")

plt.legend()

plt.show()


## Toy: buses through the day



Latent intensity rises for morning / evening peaks; observations are Poisson draws in half-hour bins.


In [ ]:
rng = np.random.default_rng(0)

X = np.linspace(0, 23.5, 48)[:, None]

intensities = 4 + 3 * np.sin(10 + 0.6 * X) + np.sin(0.1 * X)

Y = rng.poisson(intensities.ravel())[:, None]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].plot(X[:, 0], intensities[:, 0], "k-")

axes[0].set_title("True intensity")

axes[0].set_xlabel("Time (h)")

axes[1].bar(X[:, 0], Y[:, 0], width=0.45, color="0.4")

axes[1].set_title("Observed counts / 30 min")

axes[1].set_xlabel("Time (h)")

fig.tight_layout()

plt.show()


## Fit with `gpc.likelihoods.Poisson` + Laplace



An RBF prior on the log-intensity; hyperparameters are optimised by SCG on the Laplace approximate marginal.


In [ ]:
k = gpc.kern.RBF(1, variance=1.0, lengthscale=2.0)

lik = gpc.likelihoods.Poisson()

m = gpc.models.GPRegression(X, Y, kernel=k, likelihood=lik, inference="laplace")

print(m)

print("paths:", m.param_paths())

print("objective before:", m.objective())

m.optimize(iters=80, messages=True)

print("objective after:", m.objective())

print(m)


## Predictions



`predict_noiseless` returns the latent log-intensity \(f_*\). Exponentiate for the intensity scale; compare with the true curve used to generate the data.


In [ ]:
Xtest = np.linspace(0.0, 24.0, 200)[:, None]

mu_f, var_f = m.predict_noiseless(Xtest)

std = np.sqrt(var_f[:, 0])

lam = np.exp(mu_f[:, 0])

lam_lo, lam_hi = np.exp(mu_f[:, 0] - 2 * std), np.exp(mu_f[:, 0] + 2 * std)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(X[:, 0], np.log(intensities[:, 0]), "r--", lw=2, label="true log intensity")

axes[0].plot(Xtest[:, 0], mu_f[:, 0], "C0-", label=r"$\mu_f$")

axes[0].fill_between(Xtest[:, 0], mu_f[:, 0] - 2 * std, mu_f[:, 0] + 2 * std, color="C0", alpha=0.2)

axes[0].set_title("Latent log-intensity")

axes[0].set_xlabel("Time (h)")

axes[0].legend(loc="best", fontsize=8)

axes[1].bar(X[:, 0], Y[:, 0], width=0.45, color="0.75", label="counts")

axes[1].plot(X[:, 0], intensities[:, 0], "r--", lw=2, label="true intensity")

axes[1].plot(Xtest[:, 0], lam, "C0-", lw=2, label="posterior mean")

axes[1].fill_between(Xtest[:, 0], lam_lo, lam_hi, color="C0", alpha=0.2)

axes[1].set_title("Intensity vs counts")

axes[1].set_xlabel("Time (h)")

axes[1].legend(loc="best", fontsize=8)

fig.tight_layout()

plt.show()


## Bonus: Student-t (robust continuous)



Same Laplace wiring for continuous outliers — swap in `gpc.likelihoods.StudentT` (still CIP-003A). Skew / Gamma / Exponential follow the same pattern.


In [ ]:
rng = np.random.default_rng(1)

Xt = np.linspace(0.0, 1.0, 40)[:, None]

Yt = np.sin(2 * np.pi * Xt) + 0.1 * rng.standard_t(df=3, size=Xt.shape)

m_t = gpc.models.GPRegression(

    Xt,

    Yt,

    kernel="rbf",

    likelihood=gpc.likelihoods.StudentT(variance=0.25, deg_free=4.0),

    inference="laplace",

)

m_t.optimize(iters=50, messages=False)

print("Student-t objective:", m_t.objective())

mu_t, var_t = m_t.predict_noiseless(np.linspace(0, 1, 100)[:, None])

print("predictive mean range:", float(mu_t.min()), float(mu_t.max()))


## When to use which likelihood



| Likelihood | Use when |

|---|---|

| `Gaussian` (default) | Continuous \(y\); exact GP |

| `Poisson` + Laplace | Small / moderate counts; log intensity |

| `StudentT` + Laplace | Continuous \(y\) with heavy tails |

| `GPClassification` EP/Laplace | Binary labels — see [basic_classification](./basic_classification.ipynb) |



Large counts with \(\lambda \gg 1\) can often use a Gaussian GP on the counts directly (mean ≈ variance). Hetero / HMC / integral kernels stay CLI-first for now.
